# Project Aura 2.0 - Notebook 08: Monitoring & AnalyticsThis notebook sets up monitoring and analytics:- Configure structured logging via `src.utils.logging`- Track inference metrics and latency- Generate performance dashboards- Export metrics for cloud observabilityImports reusable modules from `src/` where practical.

In [ ]:
import osimport sysimport jsonimport timeimport numpy as npfrom collections import defaultdictPROJECT_DIR = "/content/project-aura"if PROJECT_DIR not in sys.path:    sys.path.insert(0, PROJECT_DIR)from src.core.config import load_configfrom src.utils.logging import get_logger, setup_logging, JsonFormatterlogger = get_logger("notebook-08")config = load_config()# Setup structured loggingsetup_logging(level=config.log_level, json_format=False)print("Logging configured")print(f"Log level: {config.log_level}")

In [ ]:
class InferenceMetrics:    """Simple metrics tracker for inference performance."""    def __init__(self):        self.latencies = []        self.predictions = defaultdict(int)        self.errors = 0        self.total_requests = 0    def record(self, latency_ms, predicted_class, success=True):        self.latencies.append(latency_ms)        self.total_requests += 1        if success:            self.predictions[predicted_class] += 1        else:            self.errors += 1    def summary(self):        if not self.latencies:            return {"error": "No data recorded"}        return {            "total_requests": self.total_requests,            "errors": self.errors,            "error_rate": self.errors / max(self.total_requests, 1),            "avg_latency_ms": float(np.mean(self.latencies)),            "p50_latency_ms": float(np.percentile(self.latencies, 50)),            "p95_latency_ms": float(np.percentile(self.latencies, 95)),            "p99_latency_ms": float(np.percentile(self.latencies, 99)),            "predictions_by_class": dict(self.predictions),        }metrics = InferenceMetrics()print("Metrics tracker initialized")

In [ ]:
np.random.seed(42)num_requests = 100for i in range(num_requests):    latency = np.random.normal(50, 10)    latency = max(latency, 5)    pred_class = np.random.randint(0, config.vision.num_classes)    success = np.random.random() > 0.05    metrics.record(latency, pred_class, success=success)print(f"Recorded {num_requests} simulated inference requests")

In [ ]:
summary = metrics.summary()print("\nInference Metrics Summary:")print(json.dumps(summary, indent=2))logger.info("Inference metrics collected", extra={    "component": "monitoring",    "metrics": summary,})

In [ ]:
metrics_dir = "/content/project-aura/artifacts"os.makedirs(metrics_dir, exist_ok=True)metrics_path = os.path.join(metrics_dir, "metrics.json")with open(metrics_path, "w") as f:    json.dump({        "timestamp": time.time(),        "component": "inference_engine",        "model": config.vision.model_name,        "metrics": summary,    }, f, indent=2)print(f"Metrics exported to: {metrics_path}")

In [ ]:
# Create a JSON formatter for cloud-native log ingestionjson_logger = get_logger("project-aura.cloud", json_format=True)json_logger.info("Inference completed", extra={    "model": config.vision.model_name,    "latency_ms": 45.2,    "predicted_class": 3,    "confidence": 0.87,    "request_id": "req-001",})json_logger.warning("High latency detected", extra={    "latency_ms": 250.5,    "threshold_ms": 100.0,    "component": "inference_engine",})print("JSON log entries emitted (visible in Colab output)")print("In production, these would be ingested by Cloud Logging")

In [ ]:
print("\n" + "="*60)print("Notebook 08 Complete - Monitoring & Analytics")print("="*60)print(f"Total requests: {summary["total_requests"]}")print(f"Average latency: {summary["avg_latency_ms"]:.2f} ms")print(f"P95 latency: {summary["p95_latency_ms"]:.2f} ms")print(f"Error rate: {summary["error_rate"]:.2%}")print(f"Metrics exported: {metrics_path}")print("\nProject Aura 2.0 notebook series complete!")print("\nNext steps:")print("  - Review PROJECT_STATUS.md for current status")print("  - Run tests: pytest tests/")print("  - Prepare for cloud deployment")